In [1]:
import subprocess
import sys
import os
from datetime import datetime

os.chdir('/workspace/PMP-514')

In [2]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path  = f'logs/train_{timestamp}.log'
os.makedirs('logs', exist_ok=True)

cmd = [
    sys.executable, 'main.py',
    '--dataset', 'bridge',
    '--gpu_id', '0',
]

print(f'Command : {" ".join(cmd)}')
print(f'Log file: {log_path}')
print('=' * 70)

with open(log_path, 'w') as log_f:
    log_f.write(f'Command: {" ".join(cmd)}\n')
    log_f.write(f'Start  : {datetime.now()}\n')
    log_f.write('=' * 70 + '\n')

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end='', flush=True)
        log_f.write(line)
        log_f.flush()

    proc.wait()

    log_f.write('=' * 70 + '\n')
    log_f.write(f'End    : {datetime.now()}\n')
    log_f.write(f'Exit   : {proc.returncode}\n')

print('=' * 70)
print(f'Exit code : {proc.returncode}')
print(f'Log saved : {log_path}')

Command : /usr/bin/python main.py --dataset bridge --gpu_id 0
Log file: logs/train_20260514_202800.log


main.py --dataset bridge --gpu_id 0                                                       | 2.64s 
--------------------------------------------------                                        | 2.64s 
                                                                                          | 2.64s 
Seed set. 1234                                                                            | 2.65s 
[Global] Dataset <bridge> Overview
	 Num Edges 9548722
	 Num Features     24
	Entire (fraud/total)   5989 / 46547 
	Train  (fraud/total)   3593 / 27928 
	Valid  (fraud/total)   1198 / 9309  
	Test   (fraud/total)   1198 / 9310  

**************** MODEL CONFIGURATION ****************
add_self_loop            -->   False
agg                      -->   mean
batch_size               -->   512
best_model_path          -->   checkpoints/
data_dir                 -->   datasets/
dataset                  -->   bridge
dataset_seed             -->   717
dropout                  -->   0.2
epochs             

In [1]:
# 셀 1: Setup — 환경 초기화 및 모델 로드
import sys, os, glob, re, json
from pathlib import Path
from argparse import Namespace
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_curve, precision_recall_curve

from utils.utils import get_config, load_data, load_checkpoint, args2config
from training_procedure import Trainer

SAVE_DIR = Path("dashboard_data")
SAVE_DIR.mkdir(exist_ok=True)

# config 재초기화
config = get_config('config/bridge.yml')['LA-SAGE-S']
config['model_name'] = 'LA-SAGE-S'
args = Namespace(
    dataset='bridge', gpu_id=0, data_dir='datasets/',
    hyper_file='config/', log_dir='logs/',
    best_model_path='checkpoints/',
    train_size=None, val_size=None, seed=None,
    num_workers=None, model='LA-SAGE-S',
    no_dev=False, run_best=False, multirun=None,
)
config = args2config(args, config)

# 최신 체크포인트 자동 탐색
ckpt_pattern = 'checkpoints/**/best_val_model_None.pth'
ckpt_files = sorted(glob.glob(ckpt_pattern, recursive=True))
assert ckpt_files, f"체크포인트를 찾을 수 없음: {ckpt_pattern}"
CKPT_PATH = ckpt_files[-1]
print(f"체크포인트: {CKPT_PATH}")

# datasetHelper 재초기화
torch.cuda.set_device(0)
datasetHelper = load_data(args, config)
datasetHelper.load()

# 모델 초기화 및 체크포인트 로드
T = Trainer(config=config, args=args, logger=None)
model, _, _, _ = T.init(datasetHelper)
model = load_checkpoint(model, CKPT_PATH)
model.eval()
print("모델 로드 완료")

체크포인트: checkpoints/val_None/LA-SAGE-S/bridge/2026-05-20_13-08-06/best_val_model_None.pth
[Global] Dataset <bridge> Overview
	 Num Edges 9548722
	 Num Features     24
	Entire (fraud/total)   5989 / 46547 
	Train  (fraud/total)   3593 / 27928 
	Valid  (fraud/total)   1198 / 9309  
	Test   (fraud/total)   1198 / 9310  

모델 로드 완료


/workspace/PMP-514/training_procedure/prepare.py:67: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at ../aten/src/ATen/native/IndexingUtils.h:27.)
  weight = (1-datasetHelper.labels[datasetHelper.train_mask]).sum() / datasetHelper.labels[datasetHelper.train_mask].sum()
/workspace/PMP-514/utils/utils.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by t

In [2]:
# 셀 2: train/val/test 전체 예측값 추출
splits = {
    'train': datasetHelper.train_loader,
    'val':   datasetHelper.val_loader,
    'test':  datasetHelper.test_loader,
}
split_nids = {
    'train': datasetHelper.train_nid.numpy(),
    'val':   datasetHelper.val_nid.numpy(),
    'test':  datasetHelper.test_nid.numpy(),
}

all_results = {}
for split_name, loader in splits.items():
    labels, fraud_probs, preds = T.evaluation(
        datasetHelper, loader, model,
        threshold_moving=True, thres=0.5
    )
    all_results[split_name] = {
        'node_ids':    split_nids[split_name],
        'labels':      labels,
        'fraud_probs': fraud_probs,
        'preds':       preds,
    }
    print(f"{split_name}: {len(labels)}개 노드, fraud_prob 범위 [{fraud_probs.min():.3f}, {fraud_probs.max():.3f}]")

print("전체 예측값 추출 완료")

/usr/local/lib/python3.11/dist-packages/dgl/dataloading/dataloader.py:1149: DGLWarning: Dataloader CPU affinity opt is not enabled, consider switching it on (see enable_cpu_affinity() or CPU best practices for DGL [https://docs.dgl.ai/tutorials/cpu/cpu_best_practises.html])
  dgl_warning(


train: 27928개 노드, fraud_prob 범위 [0.000, 1.000]
val: 9309개 노드, fraud_prob 범위 [0.000, 1.000]
test: 9310개 노드, fraud_prob 범위 [0.000, 1.000]
전체 예측값 추출 완료


In [3]:
# 셀 3: predictions.csv 저장
df_csv = pd.read_csv('datasets/final_B_bridge_sample_with_text.csv').reset_index(drop=True)
df_csv['node_id'] = df_csv.index  # node_id = CSV 행 인덱스 (preprocess_yelpzip.py 기준)

rows = []
for split_name, res in all_results.items():
    for node_id, label, fp, pred in zip(
        res['node_ids'], res['labels'], res['fraud_probs'], res['preds']
    ):
        rows.append({
            'node_id':    int(node_id),
            'fraud_prob': float(fp),
            'pred_label': int(pred),
            'split':      split_name,
        })

pred_df = pd.DataFrame(rows)
merged = df_csv.merge(pred_df, on='node_id', how='inner')

# review_month 파생 (숫자 1~12): 네트워크 페이지 groupby가 그래프 생성 로직과 일치하도록
# preprocess_yelpzip.py의 df['date'].dt.month 와 동일한 값
merged['review_month'] = merged['year_month'].str.split('-').str[1].astype(int)

# 컬럼 정렬
cols = ['node_id', 'review_id', 'user_id', 'prod_id', 'rating',
        'year_month', 'review_month', 'text', 'label', 'fraud_prob', 'pred_label', 'split']
merged = merged[[c for c in cols if c in merged.columns]]

merged.to_csv(SAVE_DIR / 'predictions.csv', index=False)
print(f"predictions.csv 저장 완료: {merged.shape}")
print(merged.groupby('split').size())

predictions.csv 저장 완료: (46547, 12)
split
test      9310
train    27928
val       9309
dtype: int64


In [4]:
# 셀 4: metrics.json 저장
val_results  = T.eval_model(all_results['val']['labels'],
                            all_results['val']['fraud_probs'],
                            all_results['val']['preds'])
test_results = T.eval_model(all_results['test']['labels'],
                            all_results['test']['fraud_probs'],
                            all_results['test']['preds'])

def to_serializable(d):
    return {k: float(v) if hasattr(v, 'item') else v for k, v in d.items()}

metrics_dict = {
    'dev':  to_serializable(val_results._asdict()),
    'test': to_serializable(test_results._asdict()),
    'config': {k: config[k] for k in
               ['epochs', 'hid_dim', 'n_layer', 'dataset', 'lr',
                'batch_size', 'patience', 'focal_loss', 'ranking_alpha']},
    'checkpoint': str(CKPT_PATH),
}

with open(SAVE_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print("metrics.json 저장 완료")
print(f"  dev  AUC: {val_results.auc_gnn:.4f}  AP: {val_results.ap_gnn:.4f}")
print(f"  test AUC: {test_results.auc_gnn:.4f}  AP: {test_results.ap_gnn:.4f}")

metrics.json 저장 완료
  dev  AUC: 0.9063  AP: 0.6780
  test AUC: 0.9059  AP: 0.6878


In [5]:
# 셀 5: roc_curve.npz + pr_curve.npz 저장
dev_fpr,  dev_tpr,  dev_roc_thres  = roc_curve(all_results['val']['labels'],  all_results['val']['fraud_probs'])
test_fpr, test_tpr, test_roc_thres = roc_curve(all_results['test']['labels'], all_results['test']['fraud_probs'])

dev_prec,  dev_rec,  dev_pr_thres  = precision_recall_curve(all_results['val']['labels'],  all_results['val']['fraud_probs'])
test_prec, test_rec, test_pr_thres = precision_recall_curve(all_results['test']['labels'], all_results['test']['fraud_probs'])

np.savez_compressed(
    SAVE_DIR / 'roc_curve.npz',
    dev_fpr=dev_fpr,   dev_tpr=dev_tpr,   dev_thresholds=dev_roc_thres,
    test_fpr=test_fpr, test_tpr=test_tpr, test_thresholds=test_roc_thres,
)
np.savez_compressed(
    SAVE_DIR / 'pr_curve.npz',
    dev_precision=dev_prec,   dev_recall=dev_rec,   dev_thresholds=dev_pr_thres,
    test_precision=test_prec, test_recall=test_rec, test_thresholds=test_pr_thres,
)

print("roc_curve.npz 저장 완료")
print("pr_curve.npz  저장 완료")

roc_curve.npz 저장 완료
pr_curve.npz  저장 완료


In [6]:
# 셀 6: embeddings_2d.npz 저장 (t-SNE)
from sklearn.manifold import TSNE

# model.relation_mlp[-1] 출력에 hook 등록 → hid_dim=40차원 임베딩 수집
emb_buffer = []

def _emb_hook(module, input, output):
    emb_buffer.append(output.detach().cpu())

handle = model.relation_mlp[-1].register_forward_hook(_emb_hook)

# test_loader: shuffle=False, worker_init_fn으로 순서 고정 → node 순서 = test_nid 순서
model.eval()
with torch.no_grad():
    for _, output_nodes, blocks in datasetHelper.test_loader:
        blocks = [b.to(torch.cuda.current_device()) for b in blocks]
        feats  = blocks[0].srcdata['feature']
        model(blocks, datasetHelper.relations, feats)

handle.remove()

embeddings = torch.cat(emb_buffer, dim=0).numpy()
print(f"임베딩 추출 완료: {embeddings.shape}")

# t-SNE 2D 변환 (수 분 소요)
print("t-SNE 변환 중...")
coords_2d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(embeddings)

np.savez_compressed(
    SAVE_DIR / 'embeddings_2d.npz',
    node_id    = all_results['test']['node_ids'],
    x          = coords_2d[:, 0],
    y          = coords_2d[:, 1],
    true_label = all_results['test']['labels'],
    fraud_prob = all_results['test']['fraud_probs'],
)
print(f"embeddings_2d.npz 저장 완료: {coords_2d.shape}")

/usr/local/lib/python3.11/dist-packages/dgl/dataloading/dataloader.py:1149: DGLWarning: Dataloader CPU affinity opt is not enabled, consider switching it on (see enable_cpu_affinity() or CPU best practices for DGL [https://docs.dgl.ai/tutorials/cpu/cpu_best_practises.html])
  dgl_warning(


임베딩 추출 완료: (9310, 40)
t-SNE 변환 중...
embeddings_2d.npz 저장 완료: (9310, 2)


In [7]:
# 셀 7: training_log.json + utu_edges.csv 저장

# ── training_log.json: 로그 파일 파싱 ─────────────────────────────────────
log_files = sorted(glob.glob('logs/train_*.log'))
LOG_PATH  = log_files[-1] if log_files else None

epoch_list, loss_list, dev_auc_list = [], [], []
if LOG_PATH:
    # 로그 형식: [Epoch X] Loss Y.YYYY | ... | Best [EX] AUC A.AAAA ...
    pattern = re.compile(
        r'\[Epoch\s+(\d+)\]\s+Loss\s+([\d.]+).*?AUC\s+([\d.]+)'
    )
    with open(LOG_PATH, encoding='utf-8') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                epoch_list.append(int(m.group(1)))
                loss_list.append(float(m.group(2)))
                dev_auc_list.append(float(m.group(3)))
    print(f"로그 파싱 완료: {LOG_PATH} ({len(epoch_list)} epochs)")
else:
    print("로그 파일 없음 — 빈 training_log.json 저장")

with open(SAVE_DIR / 'training_log.json', 'w') as f:
    json.dump({'epoch': epoch_list, 'train_loss': loss_list, 'dev_auc': dev_auc_list}, f)

# ── utu_edges.csv: DGL 그래프에서 net_utu 엣지 추출 ──────────────────────
# cosine_sim은 그래프 생성 시 저장되지 않음 → placeholder=1.0
g = datasetHelper.data
if 'net_utu' in g.etypes:
    src, dst = g.edges(etype='net_utu')
    utu_df = pd.DataFrame({
        'src':        src.numpy().astype(int),
        'dst':        dst.numpy().astype(int),
        'cosine_sim': 1.0,
    })
    utu_df.to_csv(SAVE_DIR / 'utu_edges.csv', index=False)
    print(f"utu_edges.csv 저장 완료: {len(utu_df):,} edges")
else:
    pd.DataFrame(columns=['src', 'dst', 'cosine_sim']).to_csv(
        SAVE_DIR / 'utu_edges.csv', index=False)
    print("net_utu 엣지 없음 — 빈 파일 생성")

# ── 최종 확인 ──────────────────────────────────────────────────────────────
print("\n=== dashboard_data/ 저장 완료 ===")
for f in sorted(SAVE_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<25} {size_kb:>8.1f} KB")

로그 파싱 완료: logs/train_20260520_130801.log (50 epochs)
utu_edges.csv 저장 완료: 644,230 edges

=== dashboard_data/ 저장 완료 ===
  embeddings_2d.npz            116.2 KB
  metrics.json                   1.1 KB
  pr_curve.npz                 203.9 KB
  predictions.csv            29072.2 KB
  roc_curve.npz                 22.0 KB
  training_log.json              1.0 KB
  utu_edges.csv               9789.8 KB
